# 03 Create Full Dataset Manifest

This notebook creates a full-dataset manifest for the full Renku run.

Unlike the previous pilot notebooks, this notebook does **not** sample or stratify the data. It loads all available records from the preprocessed HAM10000 and ISIC2018 manifests, ensures the lesion-size and border-touch diagnostic columns are available, concatenates both datasets, shuffles the records for processing order, and saves a single manifest.

Output:

```text
data/full_dataset/full_dataset.csv
```

Notes:

- HAM10000 contains diagnostic labels.
- ISIC2018 Task 1 provides lesion masks but no diagnostic labels, so ISIC2018 label values may be missing depending on the preprocessing manifest.
- The `mask_size_class` and `touches_any_border` columns are retained for analysis and later evaluation.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

# ============================================================
# Paths and settings

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"

HAM_CSV = DATA_DIR / "preprocessed_manifests" / "ham10000_preprocessed.csv"
ISIC_CSV = DATA_DIR / "preprocessed_manifests" / "isic2018_preprocessed.csv"

RUN_NAME = "full_dataset"
FULL_DIR = DATA_DIR / RUN_NAME
FULL_DIR.mkdir(parents=True, exist_ok=True)

FULL_CSV = FULL_DIR / f"{RUN_NAME}.csv"

RANDOM_STATE = 42
ID_COL = "stem"

print(f"BASE_DIR:  {BASE_DIR}")
print(f"DATA_DIR:  {DATA_DIR}")
print(f"RUN_NAME:  {RUN_NAME}")
print(f"FULL_DIR:  {FULL_DIR}")
print(f"FULL_CSV:  {FULL_CSV}")

BASE_DIR:  /home/jessica/Projects/DCU
DATA_DIR:  /home/jessica/Projects/DCU/data
RUN_NAME:  full_dataset
FULL_DIR:  /home/jessica/Projects/DCU/data/full_dataset
FULL_CSV:  /home/jessica/Projects/DCU/data/full_dataset/full_dataset.csv


FileNotFoundError: [Errno 2] No such file or directory: '/home/jessica/Projects/DCU/data/preprocessed_manifests/ham10000_preprocessed.csv'

In [ ]:
# Clean duplicated mask diagnostic columns after merge: duplicated columns (mask_pixels_x, mask_pixels_y) and (mask_coverage_x, mask_coverage_y)
for df_name, df in [("HAM", ham_df), ("ISIC", isic_df)]:
    if "mask_pixels_y" in df.columns:
        df["mask_pixels"] = df["mask_pixels_y"]
    elif "mask_pixels_x" in df.columns:
        df["mask_pixels"] = df["mask_pixels_x"]

    if "mask_coverage_y" in df.columns:
        df["mask_coverage"] = df["mask_coverage_y"]
    elif "mask_coverage_x" in df.columns:
        df["mask_coverage"] = df["mask_coverage_x"]

    df.drop(
        columns=[
            "mask_pixels_x",
            "mask_pixels_y",
            "mask_coverage_x",
            "mask_coverage_y",
        ],
        errors="ignore",
        inplace=True,
    )

print("HAM columns:")
print(ham_df.columns.tolist())

print("\nISIC columns:")
print(isic_df.columns.tolist())

In [ ]:

# ============================================================
# Validate common fields

required_cols = [
    "stem",
    "image_exists",
    "mask_exists",
    "img_h",
    "img_w",
    "mask_h",
    "mask_w",
    "mask_pixels",
    "mask_coverage",
    "mask_size_class",
    "touches_any_border",
    "n_borders_touched",
    "dataset",
    "lesion_id",
    "group_id",
]

def check_required_columns(df, name):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")
    print(f"{name}: all required columns present")

check_required_columns(ham_df, "HAM")
check_required_columns(isic_df, "ISIC")

assert ham_df["lesion_id"].notna().all()
assert ham_df["group_id"].notna().all()
assert isic_df["group_id"].notna().all()


In [ ]:
# ============================================================
# Ensure mask diagnostics are available for both datasets

required_mask_cols = ["mask_size_class", "touches_any_border"]
for name, df in [("HAM10000", ham_df), ("ISIC2018", isic_df)]:
    for col in required_mask_cols:
        assert col in df.columns, f"{name} manifest missing required column: {col}"

print("\nHAM mask size distribution:")
display(ham_df["mask_size_class"].value_counts(dropna=False))
print("\nHAM border-touch distribution:")
display(ham_df["touches_any_border"].value_counts(dropna=False))

print("\nISIC mask size distribution:")
display(isic_df["mask_size_class"].value_counts(dropna=False))
print("\nISIC border-touch distribution:")
display(isic_df["touches_any_border"].value_counts(dropna=False))

In [ ]:

# ============================================================
# Merge full datasets and save manifest
#
# No sampling is performed here.

full_df = (
    pd.concat([ham_df, isic_df], ignore_index=True)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

expected_total = len(ham_df) + len(isic_df)

assert len(full_df) == expected_total, f"Expected {expected_total}, got {len(full_df)}"
assert full_df[ID_COL].notna().all(), f"Missing values in ID column: {ID_COL}"
assert full_df["group_id"].notna().all(), "Missing group_id values"

# Check for duplicate dataset/stem combinations.
duplicate_count = full_df.duplicated(subset=["dataset", ID_COL]).sum()
assert duplicate_count == 0, f"Found {duplicate_count} duplicate dataset/stem records"

# HAM may legitimately contain several image rows per lesion_id.
ham_rows = full_df[full_df["dataset"] == "HAM10000"]
print("HAM image records :", len(ham_rows))
print("HAM unique lesions:", ham_rows["lesion_id"].nunique())
print(
    "HAM lesions with multiple images:",
    int((ham_rows.groupby("lesion_id").size() > 1).sum()),
)

full_df.to_csv(FULL_CSV, index=False)

print(f"Saved: {FULL_CSV}")
print("Total full-dataset records:", len(full_df))

print("\nDataset distribution:")
display(full_df["dataset"].value_counts())
display((full_df["dataset"].value_counts(normalize=True) * 100).round(2))


In [ ]:
# ============================================================
# Sanity checks and descriptive distributions

print("Full dataset allocation:")
dataset_counts = full_df["dataset"].value_counts()
display(dataset_counts)
display((dataset_counts / dataset_counts.sum() * 100).round(2))

print("\nHAM label distribution:")
ham_full = full_df[full_df["dataset"] == "HAM10000"]
display(ham_full["label"].value_counts(dropna=False).sort_index())
display((ham_full["label"].value_counts(dropna=False, normalize=True).sort_index() * 100).round(2))

print("\nHAM mask size distribution:")
display(ham_full["mask_size_class"].value_counts(dropna=False))
display((ham_full["mask_size_class"].value_counts(dropna=False, normalize=True) * 100).round(2))

print("\nHAM border-touch distribution:")
display(ham_full["touches_any_border"].value_counts(dropna=False))
display((ham_full["touches_any_border"].value_counts(dropna=False, normalize=True) * 100).round(2))

print("\nISIC mask size distribution:")
isic_full = full_df[full_df["dataset"] == "ISIC2018"]
display(isic_full["mask_size_class"].value_counts(dropna=False))
display((isic_full["mask_size_class"].value_counts(dropna=False, normalize=True) * 100).round(2))

print("\nISIC border-touch distribution:")
display(isic_full["touches_any_border"].value_counts(dropna=False))
display((isic_full["touches_any_border"].value_counts(dropna=False, normalize=True) * 100).round(2))